# Data Encryption: Implementing Encryption Techniques for Data Protection

## 📚 Learning Objectives

By completing this notebook, you will:
- Implement data encryption
- Use encryption algorithms
- Protect sensitive data
- Understand encryption types
- Apply encryption in AI systems

## 🔗 Where this fits

**Builds on:** Course 05 (AIAT 115) — Unit 2, lesson 01 "Data Loading" — data you can read from disk is data an attacker can read from disk.

---

This notebook covers practical activities from **Course 06, Unit 3**:
- Data Encryption: Implementing encryption techniques for data protection

---

## Introduction

**Data encryption** protects sensitive information by converting it into an unreadable format, essential for securing AI systems and protecting user privacy.


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- The **real Titanic passenger manifest** (`Course 04/datasets/raw/titanic.csv`).
  We encrypt and hash a genuine passenger record - a real name, ticket and cabin -
  rather than an invented string, because real identifiers are what these
  techniques exist to protect.
- `cryptography.fernet` (symmetric encryption) and `hashlib` (SHA-256).

**Outputs:** What you'll see when you run the cells

- A real record encrypted, decrypted, and verified identical.
- The SHA-256 digest of that record, and how it changes when one real field is
  altered by a single digit - integrity checking in one line.

---


In [1]:
# Concept map: name the encryption families and where each protects AI systems.
# Why first: the hands-on cell below uses all of them - this cell is the legend.

from cryptography.fernet import Fernet
import hashlib

print("✅ Libraries imported!")
print("\nData Encryption Techniques")
print("=" * 60)

# Four families, one difference that matters: who can reverse the operation
# (same key, key pair, nobody, or anyone who can compute on ciphertexts).
print("\nEncryption Types:")
print("  - Symmetric: Same key for encrypt/decrypt")
print("  - Asymmetric: Public/private key pairs")
print("  - Hashing: One-way encryption")
print("  - Homomorphic: Encrypted computation")

print("\nEncryption Algorithms:")
print("  - AES (Advanced Encryption Standard)")
print("  - RSA (Rivest-Shamir-Adleman)")
print("  - Fernet (symmetric encryption)")
print("  - SHA (Secure Hash Algorithm)")

# In AI work, encryption guards more than databases: model weights and
# sensitive features are assets worth protecting too.
print("\nApplications:")
print("  - Data at rest")
print("  - Data in transit")
print("  - Model weights")
print("  - Sensitive features")

print("\n✅ Encryption concepts understood!")

✅ Libraries imported!

Data Encryption Techniques

Encryption Types:
  - Symmetric: Same key for encrypt/decrypt
  - Asymmetric: Public/private key pairs
  - Hashing: One-way encryption
  - Homomorphic: Encrypted computation

Encryption Algorithms:
  - AES (Advanced Encryption Standard)
  - RSA (Rivest-Shamir-Adleman)
  - Fernet (symmetric encryption)
  - SHA (Secure Hash Algorithm)

Applications:
  - Data at rest
  - Data in transit
  - Model weights
  - Sensitive features

✅ Encryption concepts understood!


In [2]:
# Practice: encrypt, decrypt, and hash a REAL personal record
# WHY real data: the record below is a real passenger from the Titanic manifest.
# Its name carries a title and punctuation, and its cabin is a real code - the
# messy shapes that encryption and hashing must handle in production.

import pandas as pd
from cryptography.fernet import Fernet
import hashlib

print("Data Encryption Techniques - hands-on")
print("=" * 60)

titanic = pd.read_csv('../../../Course 04/datasets/raw/titanic.csv')  # Load: 891 real passengers
person = titanic.iloc[1]  # Pick: one real passenger with a recorded cabin
record = (f"PassengerId={person['PassengerId']};Name={person['Name']};"
          f"Age={person['Age']};Ticket={person['Ticket']};Cabin={person['Cabin']}")
print(f"\nReal record taken from the manifest ({len(titanic)} passengers):")
print(f"  {record}")

# --- 1. Symmetric encryption with Fernet (AES under the hood) ---
key = Fernet.generate_key()  # Key: one secret used to both encrypt and decrypt
cipher = Fernet(key)
token = cipher.encrypt(record.encode())  # Encrypt: plaintext -> ciphertext token
print(f"\nEncrypted: {token[:60]}...")
restored = cipher.decrypt(token).decode()  # Decrypt: only possible with the same key
print(f"Decrypted: {restored}")
assert restored == record  # Verify: encryption must be lossless
print("\u2705 Round-trip successful - only key holders can read the data.")

# --- 2. Hashing with SHA-256 (one-way!) ---
# Why hash a record you can already encrypt? To detect CHANGES: the digest is a
# fingerprint, so any edit to the record produces a completely different one.
digest = hashlib.sha256(record.encode()).hexdigest()
print(f"\nSHA-256 digest of the real record:\n  {digest}")

# Tamper with ONE real field - alter the recorded age by a single year.
old_age = f"Age={person['Age']}"
tampered = record.replace(old_age, f"Age={float(person['Age']) + 1}")
print(f"\nNow change one field ({old_age} -> Age={float(person['Age']) + 1}):")
print(f"  {hashlib.sha256(tampered.encode()).hexdigest()}")
print(f"Digests match after tampering: "
      f"{hashlib.sha256(tampered.encode()).hexdigest() == digest}")
print("\u2705 A one-character edit produces a totally different digest -> integrity checking.")
print("   Unlike encryption, hashing has NO decryption - it is one-way.")

# --- 3. Where each technique fits ---
print("\nWhere each fits in AI systems:")
print("  - Fernet/AES (symmetric): datasets and model weights at rest")
print("  - RSA (asymmetric):       exchanging keys, signing models")
print("  - SHA-256 (hashing):      integrity checks, pseudonyms (Notebook 07)")
print("  - Homomorphic (Nb. 02):   computing on encrypted data")


Data Encryption Techniques - hands-on

Real record taken from the manifest (891 passengers):
  PassengerId=2;Name=Cumings, Mrs. John Bradley (Florence Briggs Thayer);Age=38.0;Ticket=PC 17599;Cabin=C85

Encrypted: b'gAAAAABqjeAXOhZNb3LfrAkkgDDwzedhG6MQ7sjGYnBgnCc1fZ55B6sc9Hym'...
Decrypted: PassengerId=2;Name=Cumings, Mrs. John Bradley (Florence Briggs Thayer);Age=38.0;Ticket=PC 17599;Cabin=C85
✅ Round-trip successful - only key holders can read the data.

SHA-256 digest of the real record:
  55b41b3c5ecd022551214eb3d03cfba2c77c094a8aba798d7b3d1db36622a3a1

Now change one field (Age=38.0 -> Age=39.0):
  f5961f27ff8306c679037b68ea1d5b6ba614eeb6a4897ff4fa45068f79e3945a
Digests match after tampering: False
✅ A one-character edit produces a totally different digest -> integrity checking.
   Unlike encryption, hashing has NO decryption - it is one-way.

Where each fits in AI systems:
  - Fernet/AES (symmetric): datasets and model weights at rest
  - RSA (asymmetric):       exchanging keys, s

## 📚 References

1. Rivest, R. L., Shamir, A. & Adleman, L. (1978). *A Method for Obtaining Digital Signatures and Public-Key Cryptosystems*. Communications of the ACM, 21(2).
2. Daemen, J. & Rijmen, V. (2002). *The Design of Rijndael: AES — The Advanced Encryption Standard*. Springer.
3. Gentry, C. (2009). *Fully Homomorphic Encryption Using Ideal Lattices*. STOC 2009.
4. Acar, A., Aksu, H., Uluagac, A. S. & Conti, M. (2018). *A Survey on Homomorphic Encryption Schemes: Theory and Implementation*. ACM Computing Surveys, 51(4). <https://arxiv.org/abs/1704.03578>